In [1]:
#import some packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
import seaborn as sns
sns.set_style("whitegrid")
import requests
import geopandas as gdb

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression


In [2]:
# Import GIS data
from pathlib import Path

soil_folder = Path(
    r"C:\Users\lomel\Documents\GitHub\data-science\Erdos Work\GIS data\ssurgo"
)

# Find all SSURGO soil polygon shapefiles
soil_shps = list(soil_folder.rglob("soilmu_a_*.shp"))



In [3]:
# Read and combine all SSURGO soil polygons
soil_layers = [gpd.read_file(path) for path in soil_shps]

soils = gpd.GeoDataFrame(
    pd.concat(soil_layers, ignore_index=True),
    crs=soil_layers[0].crs
)

soils.shape

(27653, 5)

In [4]:
# Load HUC8 watershed boundary
gdb = r"C:\Users\lomel\Documents\GitHub\data-science\Erdos Work\GIS data\NHD_H_18060005_HU8_GDB\NHD_H_18060005_HU8_GDB.gdb"

huc8 = gpd.read_file(
    gdb,
    layer="WBDHU8"
)

# Match coordinate systems
huc8 = huc8.to_crs(soils.crs)

# Clip soils to the watershed
soils_huc8 = gpd.clip(soils, huc8)

soils_huc8.shape

(15906, 5)

In [5]:
# Find all SSURGO component tables 
comp_files = list(soil_folder.rglob("comp.txt"))

# Read and combine all SSURGO component tables
component_tables = [
    pd.read_csv(path, sep="|", header=None)
    for path in comp_files
]

components = pd.concat(
    component_tables,
    ignore_index=True
)

components.head()

,0,1,2,3,4,5,6,7,8,9,...,99,100,101,102,103,104,105,106,107,108
0,NaN,2,NaN,Linne,Series,No,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,455540,29305552
1,NaN,2,NaN,Diablo,Series,No,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,455540,29305553
2,NaN,2,NaN,Los Osos,Series,No,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,455540,29305554
3,NaN,2,NaN,Gazos,Series,No,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,455540,29305555
4,NaN,2,NaN,Millsholm,Series,No,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,455540,29305556


In [6]:
# Load SSURGO metadata
meta = pd.read_csv(
    r"C:\Users\lomel\Documents\GitHub\data-science\Erdos Work\GIS data\ssurgo\CA053\tabular\mstabcol.txt",
    sep="|",
    header=None
)

# Get the real column names for the component table
component_cols = (
    meta[meta[0] == "component"]
    .sort_values(1)[2]
    .tolist()
)

# Assign the column names
components.columns = component_cols
components.columns

Index(['comppct_l', 'comppct_r', 'comppct_h', 'compname', 'compkind',
       'majcompflag', 'otherph', 'localphase', 'slope_l', 'slope_r',
       ...
       'flsoilleachpot', 'flsoirunoffpot', 'fltemik2use', 'fltriumph2use',
       'indraingrp', 'innitrateleachi', 'misoimgmtgrp', 'vasoimgtgrp', 'mukey',
       'cokey'],
      dtype='str', length=109)

In [7]:
# Get only the MUKEY values that are inside the HUC8
huc8_mukeys = soils_huc8["MUKEY"].astype(str).unique()

# Keep only components belonging to soil map units inside the HUC8
components_huc8 = components[
    components["mukey"].astype(str).isin(huc8_mukeys)
]

components_huc8.shape

(3074, 109)

In [8]:
# Find all SSURGO map-unit aggregate tables
muaggatt_files = list(soil_folder.rglob("muaggatt.txt"))

# Read and combine SSURGO map-unit aggregate tables
muaggatt_tables = [
    pd.read_csv(path, sep="|", header=None)
    for path in muaggatt_files
]

muaggatt = pd.concat(
    muaggatt_tables,
    ignore_index=True
)

In [9]:
muaggatt_cols = (
    meta[meta[0] == "muaggatt"]
    .sort_values(1)[2]
    .tolist()
)

muaggatt.columns = muaggatt_cols

In [10]:
muaggatt_huc8 = muaggatt[
    muaggatt["mukey"].astype(str).isin(huc8_mukeys)
]

muaggatt_huc8.shape

(492, 40)

In [11]:
muaggatt_huc8.columns.tolist()

['musym',
 'muname',
 'mustatus',
 'slopegraddcp',
 'slopegradwta',
 'brockdepmin',
 'wtdepannmin',
 'wtdepaprjunmin',
 'flodfreqdcd',
 'flodfreqmax',
 'pondfreqprs',
 'aws025wta',
 'aws050wta',
 'aws0100wta',
 'aws0150wta',
 'drclassdcd',
 'drclasswettest',
 'hydgrpdcd',
 'iccdcd',
 'iccdcdpct',
 'niccdcd',
 'niccdcdpct',
 'engdwobdcd',
 'engdwbdcd',
 'engdwbll',
 'engdwbml',
 'engstafdcd',
 'engstafll',
 'engstafml',
 'engsldcd',
 'engsldcp',
 'englrsdcd',
 'engcmssdcd',
 'engcmssmp',
 'urbrecptdcd',
 'urbrecptwta',
 'forpehrtdcp',
 'hydclprs',
 'awmmfpwwta',
 'mukey']

In [12]:
# Select soil variables relevant to nitrate transport and hydrology
soil_vars = [
    "mukey",
    "slopegradwta",
    "brockdepmin",
    "wtdepannmin",
    "aws0150wta",
    "drclassdcd",
    "hydgrpdcd",
    "flodfreqdcd"
]

soils_attributes = muaggatt_huc8[soil_vars]

soils_attributes.head()

,mukey,slopegradwta,brockdepmin,wtdepannmin,aws0150wta,drclassdcd,hydgrpdcd,flodfreqdcd
0,455537,6.0,91.0,NaN,14.56,Well drained,D,NaN
1,455538,12.0,91.0,NaN,14.56,Well drained,D,NaN
2,455539,23.0,91.0,NaN,14.56,Well drained,D,NaN
3,455540,40.0,91.0,NaN,14.56,Well drained,D,NaN
4,455541,38.0,43.0,NaN,12.00,Well drained,D,NaN


In [13]:
soils_attributes.isna().sum()

mukey             0
slopegradwta      1
brockdepmin     234
wtdepannmin     468
aws0150wta       16
drclassdcd       31
hydgrpdcd        23
flodfreqdcd     458
dtype: int64

In [14]:
# Keep soil predictors with good data coverage
soil_vars_final = [
    "mukey",# mukey = unique ID for each SSURGO soil map unit
    "slopegradwta",# slopegradwta = weighted average slope gradient for the map unit
    "brockdepmin", # brockdepmin = minimum depth to bedrock within the map unit
    "aws0150wta", # aws0150wta = weighted average available water storage in the top 150 cm of soil
    "drclassdcd",# drclassdcd = dominant drainage class of the map unit
    "hydgrpdcd"# hydgrpdcd = dominant hydrologic soil group of the map unit
]
soils_attributes = muaggatt_huc8[soil_vars_final]

In [15]:
# Make MUKEY the same data type in both tables
soils_huc8["MUKEY"] = soils_huc8["MUKEY"].astype(str)
soils_attributes["mukey"] = soils_attributes["mukey"].astype(str)

# Join soil attributes to the HUC8 soil polygons
soils_model = soils_huc8.merge(
    soils_attributes,
    left_on="MUKEY",
    right_on="mukey",
    how="left"
)

soils_model.shape

(15906, 11)

In [16]:
# Remove duplicate join key
soils_model = soils_model.drop(columns=["mukey"])

In [17]:
# Save final soil layer with selected attributes
soils_model.to_file(
    r"C:\Users\lomel\Documents\GitHub\data-science\Erdos Work\GIS data\soils_huc8_attributes.gpkg",
    layer="soils_huc8_attributes",
    driver="GPKG"
)

In [18]:
soils_model.shape
soils_model.head()

,AREASYMBOL,SPATIALVER,MUSYM,MUKEY,geometry,slopegradwta,brockdepmin,aws0150wta,drclassdcd,hydgrpdcd
0,CA772,4,32,465299,"POLYGON ((-120.26516 35.22769, -120.26465 35.2...",45.0,76.0,10.56,Well drained,B
1,CA772,4,23,465289,"POLYGON ((-120.21429 35.2328, -120.21514 35.23...",30.0,89.0,14.52,Well drained,D
2,CA772,4,25,465291,"POLYGON ((-120.41188 35.2657, -120.41193 35.26...",45.0,41.0,7.60,Well drained,D
3,CA772,4,50,465319,"POLYGON ((-120.35982 35.28844, -120.35848 35.2...",8.0,NaN,17.58,Well drained,A
4,CA772,4,31,465298,"POLYGON ((-120.27662 35.3126, -120.27676 35.31...",35.0,41.0,13.98,Well drained,C


In [19]:
# 1. Initial profiling
print(soils_model.shape)
soils_model.info()
soils_model.describe(include="all").T

(15906, 10)
<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 15906 entries, 0 to 15905
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype   
---  ------        --------------  -----   
 0   AREASYMBOL    15906 non-null  str     
 1   SPATIALVER    15906 non-null  int32   
 2   MUSYM         15906 non-null  str     
 3   MUKEY         15906 non-null  str     
 4   geometry      15906 non-null  geometry
 5   slopegradwta  15898 non-null  float64 
 6   brockdepmin   8996 non-null   float64 
 7   aws0150wta    15251 non-null  float64 
 8   drclassdcd    14312 non-null  str     
 9   hydgrpdcd     14784 non-null  str     
dtypes: float64(3), geometry(1), int32(1), str(5)
memory usage: 1.6 MB


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
AREASYMBOL,15906,5,CA053,9589,NaN,NaN,NaN,NaN,NaN,NaN,NaN
SPATIALVER,15906.0,NaN,NaN,NaN,9.215956,2.374696,4.0,6.0,11.0,11.0,11.0
MUSYM,15906,417,Rc,352,NaN,NaN,NaN,NaN,NaN,NaN,NaN
MUKEY,15906,492,3251916,352,NaN,NaN,NaN,NaN,NaN,NaN,NaN
geometry,15906,15906,"POLYGON ((-120.265156973946 35.2276879730751, ...",1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
slopegradwta,15898.0,NaN,NaN,NaN,26.703611,20.591332,0.0,6.0,23.0,40.0,140.3
brockdepmin,8996.0,NaN,NaN,NaN,63.818586,29.774789,0.0,41.0,71.0,79.0,173.0
aws0150wta,15251.0,NaN,NaN,NaN,14.267751,6.419165,2.25,9.94,14.22,19.5,31.5
drclassdcd,14312,7,Well drained,12404,NaN,NaN,NaN,NaN,NaN,NaN,NaN
hydgrpdcd,14784,5,C,7374,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [20]:
# Check missing values at the unique soil map-unit level
soil_units = (
    soils_model
    .drop_duplicates("MUKEY")
)

soil_units[
    ["slopegradwta", "brockdepmin", "aws0150wta",
     "drclassdcd", "hydgrpdcd"]
].isna().sum()

slopegradwta      1
brockdepmin     234
aws0150wta       16
drclassdcd       31
hydgrpdcd        23
dtype: int64

In [21]:
# Check duplicate geometries / rows
soils_model.duplicated().sum()

np.int64(0)

In [22]:
# Check invalid geometries
(~soils_model.geometry.is_valid).sum()

np.int64(0)